# Chapter 18: Fine-Tuning Locally (Hugging Face + Unsloth)
## Topic 1: Prompting vs Fine-Tuning — When Each Actually Makes Sense

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Chapter 8 Topic 8 already asked this exact question once — RAG vs fine-tuning — and concluded RAG's simple re-ingest mechanism beats fine-tuning specifically for *knowledge-freshness* problems (Chapter 9 Topic 8's Smart Saver FD proof made this concrete). This topic asks the same underlying question again, but from a genuinely different angle: not "should we fine-tune instead of building RAG," but "given everything this notebook series has already built (RAG, tools, memory, agentic retrieval), does fine-tuning still have a legitimate, complementary role, and if so, what specifically is it good for that prompting alone cannot achieve?"
- The core distinction this topic draws precisely: prompting (including RAG, tool-calling, and every technique built through Chapter 17) changes what information and instructions a model sees *at inference time*, leaving the model's own underlying weights completely untouched. Fine-tuning changes the model's actual weights, permanently altering how it responds to a whole category of input, without needing that instruction repeated in every single prompt. These are not competing solutions to the same problem — they modify genuinely different things, and recognizing this distinction is what makes the "when to use which" question answerable at all.
- The specific class of problem fine-tuning is well-suited for, precisely because it differs from RAG's knowledge-freshness strength: **behavioral and stylistic consistency that would otherwise require an increasingly long, repeated prompt** — for this project's specific context, teaching the model a very particular way of handling Hinglish phrasing patterns, a specific tone for regulated-domain customer communication, or a very particular reasoning style for ambiguous Multiple Category cases, all *without* needing every one of these instructions spelled out at length in the system prompt on every single call.


### 2. Internal Working — Step by Step

**The concrete decision process for choosing between prompting and fine-tuning, step by step:**

1. **Ask whether the problem is fundamentally about the model lacking access to specific, changeable information** — if yes, this is Chapter 8 Topic 8's and Chapter 9 Topic 8's territory: RAG (or a tool call) is the right answer, since fine-tuning bakes information into frozen weights that becomes stale exactly as fast as the underlying facts change, while RAG's knowledge base can be updated instantly through re-ingestion (Chapter 9 Topic 8's own demonstrated mechanism).
2. **Ask whether the problem is fundamentally about the model's behavior, style, or reasoning pattern needing to change consistently, across many different inputs, in a way that's hard to fully specify through prompting alone** — if yes, this is fine-tuning's actual, distinct strength: baking a consistent behavioral pattern into the model's weights so it doesn't need to be re-explained, in full detail, in every single prompt.
3. **Weigh the very different cost and iteration-speed profiles of the two approaches** — a prompt change (Chapter 10 Topic 4's validation discipline) can typically be tested and deployed within minutes to hours; a fine-tuning run (this chapter's remaining topics) requires assembling a training dataset (Topic 2), running an actual training process (Topics 3-6), and validating the result (Topic 7) — a genuinely longer, more resource-intensive iteration cycle that should only be undertaken once prompting has been given a genuine, evidence-based chance to solve the problem first.
4. **Recognize that the two approaches can be combined, not just chosen between** — a fine-tuned model can still use RAG for knowledge-freshness needs and tool-calling for structured actions; fine-tuning specifically targets the model's baseline behavioral tendencies, while prompting and retrieval continue to handle everything they're already well-suited for on top of that fine-tuned baseline.


### 3. How This Is Implemented in This Project

- This project's entire GenAI pipeline, built from Chapter 2 through Chapter 17, has exclusively used prompting-based techniques (including RAG, Chapters 7-9, and tool-calling, Chapter 10) — this project has not needed fine-tuning at all up to this point, which is itself informative: the specific problems this project has actually faced (classification accuracy, retrieval quality, hallucination) have all been addressable through prompting-based techniques, exactly matching this topic's own decision criterion that fine-tuning should only be pursued once prompting has had a genuine chance.
- Chapter 16's own real error-analysis findings are the concrete, evidence-based starting point for genuinely evaluating whether fine-tuning is now warranted: Topic 3's demonstrated finding that this project's dominant real failure pattern (Hinglish-heavy phrasing) required a *structural* fix (query-side term expansion), not a prompting fix, is directly relevant here — if a similar, careful investigation revealed a *behavioral* pattern (not an information-access pattern) that prompting genuinely couldn't address consistently, that would be the concrete, evidence-based trigger for considering fine-tuning specifically.
- This chapter's stated dataset-preparation approach (Topic 2's "hard-case rows" from this project's own real CSV data) directly continues Chapter 16's error-analysis discipline — using the project's own real, identified difficult cases as the specific training signal for fine-tuning, rather than fine-tuning on a generic, unrelated dataset disconnected from this project's actual, demonstrated needs.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Fine-tuning without first exhausting prompting-based approaches is a common, costly mistake this notebook series' own evidence-based adoption discipline argues directly against** — mirroring exactly Chapter 6's vector-database migration trigger, Chapter 11 Topic 3's MCP adoption trigger, and Chapter 16 Topic 3's prompting-vs-structural-fix diagnostic: a larger, more expensive intervention (fine-tuning) should only be pursued once genuine evidence confirms a cheaper, faster alternative (prompting) genuinely cannot address the problem, not adopted preemptively based on fine-tuning's reputation as a more sophisticated technique.
- **A fine-tuned model's behavioral change is genuinely harder to iterate on and reverse than a prompt change** — a bad prompt change can be reverted in minutes; a bad fine-tuning run requires re-preparing data, re-running training, and re-validating before a corrected version is available, a real, meaningfully slower iteration cycle that should factor into the decision of when fine-tuning is actually worth this cost.
- **Fine-tuning a model on a specific behavioral pattern risks unintended side effects on the model's behavior for inputs outside that specific pattern** — a phenomenon sometimes called catastrophic forgetting or capability regression, where fine-tuning for one specific improvement inadvertently degrades performance on other, previously-well-handled cases — this needs the same evidence-based before/after validation (Topic 7) as any other pipeline change in this notebook series, checking not just whether the targeted behavior improved, but whether anything else regressed.
- **This project's specific hardware constraint (a single RTX 4060 with 8GB VRAM) rules out full fine-tuning of most modern LLMs outright** — full fine-tuning requires storing and updating every one of a model's parameters, which for even a comparatively small 8-billion-parameter model vastly exceeds 8GB of available memory; this hardware reality is precisely why this chapter's remaining topics (PEFT, LoRA, QLoRA) exist as the practical path to fine-tuning at all on this specific, real hardware target.
- **Monitoring:** if fine-tuning is adopted, this project's existing Chapter 15 regression-tracking discipline needs to extend to the fine-tuned model as its own explicitly versioned artifact, distinct from prompt versions and agent-graph versions — a fine-tuned model checkpoint is a new kind of "version" this project's evaluation harness needs to account for.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Attempting a genuinely thorough prompting solution first vs moving to fine-tuning quickly once a behavioral pattern is suspected:** given fine-tuning's meaningfully higher cost and slower iteration cycle, a genuine, evidence-based prompting attempt (following Chapter 16 Topic 3's own diagnostic discipline) should always come first — fine-tuning is the more expensive, slower-to-iterate option, and should be reserved for cases where prompting has been given a real, fair chance and genuinely falls short.
- **Fine-tuning for a narrow, specific behavioral pattern vs a broader, more general behavioral shift:** a narrow, targeted fine-tuning objective (Topic 2's "hard-case rows" approach) is more tractable to prepare data for, validate, and reason about potential side effects on — a broader, more general fine-tuning objective is harder to validate comprehensively and carries a correspondingly larger risk of unintended capability regression elsewhere.
- **Whether fine-tuning should replace or complement this project's existing prompting-based pipeline:** complementing (this topic's recommended framing) — a fine-tuned model still benefits from RAG for knowledge-freshness needs and tool-calling for structured actions; fine-tuning specifically addresses the model's baseline behavioral tendencies, not a wholesale replacement of everything this notebook series has already built.


### 6. Alternatives and When to Use Each

- **Prompting-based techniques alone (this project's exclusive approach through Chapter 17):** the right, sufficient choice for information-access problems (RAG's specific strength) and for behavioral adjustments that can be adequately and consistently specified through prompt instructions.
- **Fine-tuning, layered on top of an otherwise prompting-based pipeline (this chapter's actual subject):** worth adopting specifically once genuine evidence (following Chapter 16 Topic 3's diagnostic discipline) shows a behavioral or stylistic pattern that prompting genuinely cannot address consistently, not adopted preemptively or by default.
- **Full fine-tuning of every model parameter:** ruled out directly by this project's actual hardware constraint (a single RTX 4060 with 8GB VRAM) — this chapter's remaining topics (PEFT, LoRA, QLoRA) exist specifically as the practical alternative given this real constraint.


### 7. Common Mistakes and Production Failures

- Reaching for fine-tuning before genuinely exhausting prompting-based approaches, incurring fine-tuning's real, higher cost and slower iteration cycle for a problem a cheaper prompt change could have solved.
- Conflating an information-access problem (RAG's strength) with a behavioral-consistency problem (fine-tuning's strength), applying the wrong category of solution to the actual underlying need.
- Not validating a fine-tuned model for potential capability regression on cases outside its specific targeted improvement, risking an unnoticed net-negative change despite the targeted improvement succeeding.
- Assuming full fine-tuning is feasible without accounting for this project's actual, real hardware constraint (a single RTX 4060, 8GB VRAM), which rules out full parameter fine-tuning for essentially any modern LLM of meaningful size.
- Treating fine-tuning as a wholesale replacement for this project's existing prompting-based pipeline, rather than a complementary technique addressing a specific, different category of need.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What's the fundamental difference between what prompting and fine-tuning each change?
  A: Prompting changes what information and instructions a model sees at inference time, leaving its underlying weights completely untouched. Fine-tuning changes the model's actual weights, permanently altering its behavior for a whole category of input without needing that instruction repeated in every prompt.

- Q: Why does this project's real hardware constraint (a single RTX 4060, 8GB VRAM) rule out full fine-tuning?
  A: Full fine-tuning requires storing and updating every one of a model's parameters during training, which for even a comparatively small multi-billion-parameter model vastly exceeds 8GB of available VRAM — this hardware reality is precisely why this chapter's remaining topics on PEFT, LoRA, and QLoRA exist as the practical path to fine-tuning at all on this specific hardware.

**Intermediate**

- Q: How does this topic's prompting-vs-fine-tuning question differ from Chapter 8 Topic 8's earlier RAG-vs-fine-tuning discussion?
  A: Chapter 8 Topic 8 specifically established that fine-tuning is poorly suited to knowledge-freshness problems compared to RAG's simple re-ingest mechanism. This topic asks a broader, complementary question: given that RAG, tools, and every other prompting-based technique this notebook series has built already address information-access and structured-action needs, does fine-tuning still have a legitimate, different role — specifically for behavioral and stylistic consistency that would otherwise require an increasingly long, repeated prompt.

- Q: Why should a genuine prompting attempt always precede a decision to fine-tune?
  A: Fine-tuning has a meaningfully higher cost and slower iteration cycle than a prompt change — a bad prompt can be reverted in minutes, while a bad fine-tuning run requires re-preparing data, re-running training, and re-validating. Given this cost asymmetry, prompting should be given a genuine, evidence-based chance first, mirroring exactly the same evidence-based adoption discipline already established for other larger architectural decisions in this notebook series (Chapter 6's vector-database migration, Chapter 11 Topic 3's MCP adoption).

**Advanced**

- Q: Using Chapter 16's real error-analysis findings, walk through how you'd decide whether this project's current failure patterns actually warrant fine-tuning.
  A: Chapter 16 Topic 3 already demonstrated concretely that this project's dominant real failure pattern (Hinglish-heavy phrasing) required a structural, retrieval-layer fix (query-side term expansion), not a prompting fix and not a behavioral, fine-tuning-addressable pattern — this specific finding argues against fine-tuning as the right response to that particular pattern. Fine-tuning would only become genuinely warranted if a *different*, specifically behavioral pattern were identified through this same rigorous error-analysis process — one where the model's actual reasoning style or response tendency, not its access to information, was the demonstrated root cause, and where a genuine, evidence-based prompting attempt at addressing that behavioral pattern had already been tried and found insufficient.

- Q: A teammate proposes fine-tuning this project's model to "generally improve its handling of Hinglish content," given that this is the project's dominant real failure pattern. What's your concern given this topic's framework?
  A: Chapter 16 Topic 3 already established, with real, measured evidence, that this specific failure pattern's root cause is a retrieval-layer vocabulary-mismatch problem (BM25 failing to match Hinglish query terms against English-only knowledge base content), not a behavioral or reasoning problem in the model itself — no amount of fine-tuning the model's own weights would fix a case where the correct information was never even retrieved in the first place. Fine-tuning would be solving the wrong layer of the actual, already-diagnosed problem; the correct fix (already validated) is the structural, retrieval-layer term-expansion technique, not any change to the model's own weights.

**Scenario-based**

- Q: Chapter 16's error analysis reveals a new pattern: the model consistently uses an overly informal, inconsistent tone when responding to sensitive, regulated-domain topics, despite the system prompt explicitly specifying a formal, consistent tone. Prompting refinements have been tried repeatedly without success. Walk through your decision process.
  A: This is a genuinely different kind of pattern than the Hinglish case — it's not about missing information (RAG's domain) but about a consistent behavioral/stylistic tendency the model exhibits despite explicit, repeated prompting instructions, and prompting has already been given a genuine, evidence-based chance and found insufficient. This is exactly the profile Topic 1's framework identifies as fine-tuning's legitimate, distinct strength — baking a specific, consistent tonal and stylistic pattern into the model's weights, so it doesn't depend on the prompt instruction being perfectly followed every single time. This would be a reasonable candidate to proceed with this chapter's remaining fine-tuning topics for.

**Follow-up questions to expect:**

- "How would you design an evaluation to specifically confirm a failure pattern is behavioral rather than information-access-related, before committing to fine-tuning?"
- "What would you do if, partway through preparing a fine-tuning dataset, you discovered the pattern was actually more information-access-related than you initially thought?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **This topic's decision framework — "does this need new information, or a new behavioral tendency" — is a specific instance of a much more general engineering question**: distinguishing a data/knowledge problem from a capability/behavior problem, a distinction relevant to human skill development and organizational process design as much as to machine learning systems.
- **The prompting-vs-fine-tuning cost-and-iteration-speed trade-off directly echoes this notebook series' repeated evidence-based-adoption principle** — apply a cheaper, faster-to-validate intervention first, and only escalate to a more expensive, slower-to-iterate one once genuine evidence confirms the cheaper option is insufficient — exactly the same discipline already established for MCP adoption (Chapter 11 Topic 3), vector-database migration (Chapter 6), and prompting-vs-structural fixes (Chapter 16 Topic 3).
- **This topic's conclusion — that fine-tuning is warranted specifically for genuine, prompting-resistant behavioral patterns — is the necessary justification for the rest of this chapter's real engineering investment**: Topics 2-8's dataset preparation, actual training mechanics, and evaluation work only make sense once this topic's diagnostic question has actually been answered affirmatively for a specific, real, identified need.

### 10. Quick Revision Summary (for last-minute interview prep)

> Prompting and fine-tuning modify genuinely different things: prompting changes what a model sees at inference time, leaving its weights untouched; fine-tuning changes the model's actual weights, baking in a consistent behavioral change that doesn't need to be re-specified in every prompt. This makes them suited to different problem categories, not competing solutions to the same problem — information-access and knowledge-freshness needs (Chapter 8 Topic 8, Chapter 9 Topic 8) are RAG's strength, since fine-tuning bakes information into weights that go stale as fast as the underlying facts change; consistent behavioral or stylistic patterns that are hard to fully specify through prompting are fine-tuning's distinct strength. Given fine-tuning's genuinely higher cost and slower iteration cycle compared to a prompt change, a real, evidence-based prompting attempt should always be given a fair chance first, mirroring this notebook series' repeated evidence-based-adoption discipline (Chapter 6, Chapter 11 Topic 3, Chapter 16 Topic 3) — Chapter 16 Topic 3's own real finding that this project's dominant failure pattern needed a structural (not fine-tuning-addressable) fix is a direct, concrete illustration of this diagnostic principle in action. This project's actual hardware constraint (a single RTX 4060, 8GB VRAM) rules out full fine-tuning entirely, making the parameter-efficient techniques (PEFT, LoRA, QLoRA) this chapter's remaining topics build out not just a convenient choice but the only practically feasible path to fine-tuning at all on this specific, real target hardware.


### Module 1: A Real Diagnostic Function — Information-Access vs Behavioral Pattern

Build an executable decision function applying this topic's framework, tested against Chapter 16's own real, already-diagnosed findings.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Real diagnostic decision function
# ------------------------------------------------------------------

def diagnose_fix_category(pattern_description: str, requires_external_info: bool,
                            prompting_already_attempted: bool, prompting_succeeded: bool) -> str:
    """A REAL, executable decision function applying this topic's
    framework -- not just a described principle, but actual logic
    that can be run against real, identified failure patterns."""
    if requires_external_info:
        return "RAG / STRUCTURAL FIX -- information-access problem, not fixable by fine-tuning"

    if not prompting_already_attempted:
        return "TRY PROMPTING FIRST -- behavioral pattern, but prompting hasn't been genuinely attempted yet"

    if prompting_already_attempted and prompting_succeeded:
        return "PROMPTING SUFFICIENT -- no need to fine-tune"

    return "FINE-TUNING CANDIDATE -- behavioral pattern, prompting genuinely attempted and insufficient"


# REAL cases: Chapter 16's own actual, already-diagnosed finding, plus
# a genuinely different, hypothetical BEHAVIORAL pattern for contrast
REAL_CASES = [
    {
        "pattern": "Hinglish-heavy phrasing causing BM25 retrieval failures (Chapter 16 Topic 3's REAL finding)",
        "requires_external_info": True,  # retrieval/vocabulary-matching problem
        "prompting_already_attempted": True,
        "prompting_succeeded": False,  # Chapter 16 Topic 3 measured 0% improvement from prompting
    },
    {
        "pattern": "Inconsistent, overly informal tone on sensitive regulated-domain topics",
        "requires_external_info": False,  # NOT an information problem
        "prompting_already_attempted": True,
        "prompting_succeeded": False,  # hypothetically, repeated prompting attempts failed
    },
]

print("=" * 70)
print("MODULE 1: REAL DIAGNOSTIC DECISIONS ON ACTUAL PROJECT FINDINGS")
print("=" * 70)

for case in REAL_CASES:
    decision = diagnose_fix_category(case["pattern"], case["requires_external_info"],
                                       case["prompting_already_attempted"], case["prompting_succeeded"])
    case_pattern = case["pattern"]
    print(f"\nPattern: {case_pattern}")
    print(f"  DECISION: {decision}")

print("\nCONFIRMED: applying this topic's diagnostic framework to Chapter 16's")
print("OWN real finding correctly routes it to RAG/structural fix (NOT")
print("fine-tuning) -- consistent with what was ACTUALLY validated there with")
print("real, measured data. The hypothetical tone-consistency pattern correctly")
print("routes to FINE-TUNING CANDIDATE, illustrating the genuinely different")
print("category of problem this chapter's techniques are meant to address.")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: REAL DIAGNOSTIC DECISIONS ON ACTUAL PROJECT FINDINGS

Pattern: Hinglish-heavy phrasing causing BM25 retrieval failures (Chapter 16 Topic 3's REAL finding)
  DECISION: RAG / STRUCTURAL FIX -- information-access problem, not fixable by fine-tuning

Pattern: Inconsistent, overly informal tone on sensitive regulated-domain topics
  DECISION: FINE-TUNING CANDIDATE -- behavioral pattern, prompting genuinely attempted and insufficient

CONFIRMED: applying this topic's diagnostic framework to Chapter 16's
OWN real finding correctly routes it to RAG/structural fix (NOT
fine-tuning) -- consistent with what was ACTUALLY validated there with
real, measured data. The hypothetical tone-consistency pattern correctly
routes to FINE-TUNING CANDIDATE, illustrating the genuinely different
category of problem this chapter's techniques are meant to address.

Module 1 complete. Run Module 2 next.


### Module 2: Real Memory Math — Why Full Fine-Tuning Doesn't Fit on an 8GB GPU

Compute the actual, real memory footprint of full fine-tuning for an 8-billion-parameter model, showing concretely why it exceeds this project's real RTX 4060's 8GB VRAM.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Real memory math for full fine-tuning feasibility
# ------------------------------------------------------------------

# REAL, standard memory accounting for full fine-tuning in mixed
# precision (bf16/fp16) with the Adam optimizer -- this is the actual,
# well-established formula used throughout ML systems engineering
MODEL_PARAMS_BILLIONS = 8.0  # Llama-3.1-8B-Instruct, this project's actual target model
BYTES_PER_PARAM_WEIGHTS_FP16 = 2       # model weights, stored in fp16/bf16
BYTES_PER_PARAM_GRADIENTS_FP16 = 2     # gradients, same precision as weights
BYTES_PER_PARAM_OPTIMIZER_STATES = 8   # Adam optimizer: 2 states (momentum + variance), fp32 each = 4+4 bytes

RTX_4060_VRAM_GB = 8.0  # this project's REAL, actual target hardware

def compute_full_finetuning_memory_gb(params_billions: float) -> dict:
    """REAL, standard memory footprint calculation for full fine-tuning
    -- weights + gradients + optimizer states, the actual components
    that must ALL fit in GPU memory simultaneously during training."""
    n_params = params_billions * 1e9
    weights_gb = (n_params * BYTES_PER_PARAM_WEIGHTS_FP16) / 1e9
    gradients_gb = (n_params * BYTES_PER_PARAM_GRADIENTS_FP16) / 1e9
    optimizer_gb = (n_params * BYTES_PER_PARAM_OPTIMIZER_STATES) / 1e9
    total_gb = weights_gb + gradients_gb + optimizer_gb
    return {"weights_gb": weights_gb, "gradients_gb": gradients_gb,
            "optimizer_gb": optimizer_gb, "total_gb": total_gb}


memory_breakdown = compute_full_finetuning_memory_gb(MODEL_PARAMS_BILLIONS)

print("=" * 70)
print("MODULE 2: REAL MEMORY MATH -- FULL FINE-TUNING FEASIBILITY")
print("=" * 70)
print(f"\nModel: Llama-3.1-8B-Instruct ({MODEL_PARAMS_BILLIONS}B parameters) -- this project's REAL target")
print(f"\nFull fine-tuning memory requirements (standard, real formula):")
w_gb = memory_breakdown["weights_gb"]
g_gb = memory_breakdown["gradients_gb"]
o_gb = memory_breakdown["optimizer_gb"]
t_gb = memory_breakdown["total_gb"]
print(f"  Model weights (fp16):        {w_gb:.1f} GB")
print(f"  Gradients (fp16):            {g_gb:.1f} GB")
print(f"  Optimizer states (Adam):     {o_gb:.1f} GB")
print(f"  TOTAL required:              {t_gb:.1f} GB")

print(f"\nThis project's REAL available VRAM (RTX 4060): {RTX_4060_VRAM_GB} GB")

deficit = memory_breakdown["total_gb"] - RTX_4060_VRAM_GB
print(f"\nMEMORY DEFICIT: {deficit:.1f} GB short of what full fine-tuning requires")
print(f"(this doesn't even include activation memory, which adds MORE on top)")

feasible = memory_breakdown["total_gb"] <= RTX_4060_VRAM_GB
print(f"\nIs full fine-tuning feasible on this hardware? {feasible}")

if not feasible:
    ratio = memory_breakdown["total_gb"] / RTX_4060_VRAM_GB
    print(f"\nCONFIRMED: full fine-tuning requires {ratio:.1f}x more memory than this")
    print(f"project's actual available VRAM -- a REAL, computed, unambiguous")
    print(f"demonstration of why full fine-tuning is NOT an option here, and why")
    print(f"this chapter's remaining topics (PEFT, LoRA, QLoRA) exist as the")
    print(f"only practically feasible path to fine-tuning on this real hardware.")

print("\nModule 2 complete. All Chapter 18 Topic 1 modules done.")
print()
print("=" * 70)
print("CHAPTER 18 TOPIC 1 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  A REAL diagnostic function correctly routed Chapter 16's OWN actual
  finding (Hinglish phrasing) to "RAG/structural fix, NOT fine-tuning"
  -- consistent with what was genuinely validated there with real data.

  REAL memory math: full fine-tuning of an 8B model requires ~96 GB
  (weights + gradients + optimizer states) -- computed using the
  actual, standard formula used in ML systems engineering, not an
  assumed or hand-waved number.

  This 96 GB requirement is ~12x this project's REAL available VRAM
  (8 GB, RTX 4060) -- a concrete, unambiguous, COMPUTED demonstration
  of why full fine-tuning is infeasible here, motivating this chapter's
  entire remaining arc (PEFT, LoRA, QLoRA).

  Fine-tuning should be reserved for GENUINE behavioral patterns where
  prompting has been fairly attempted and found insufficient -- never
  for information-access problems (RAG's domain) and never adopted
  preemptively before prompting has had a real, evidence-based chance.
""")


MODULE 2: REAL MEMORY MATH -- FULL FINE-TUNING FEASIBILITY

Model: Llama-3.1-8B-Instruct (8.0B parameters) -- this project's REAL target

Full fine-tuning memory requirements (standard, real formula):
  Model weights (fp16):        16.0 GB
  Gradients (fp16):            16.0 GB
  Optimizer states (Adam):     64.0 GB
  TOTAL required:              96.0 GB

This project's REAL available VRAM (RTX 4060): 8.0 GB

MEMORY DEFICIT: 88.0 GB short of what full fine-tuning requires
(this doesn't even include activation memory, which adds MORE on top)

Is full fine-tuning feasible on this hardware? False

CONFIRMED: full fine-tuning requires 12.0x more memory than this
project's actual available VRAM -- a REAL, computed, unambiguous
demonstration of why full fine-tuning is NOT an option here, and why
this chapter's remaining topics (PEFT, LoRA, QLoRA) exist as the
only practically feasible path to fine-tuning on this real hardware.

Module 2 complete. All Chapter 18 Topic 1 modules done.

CHAPTER